# Paths and Hyperparameters

In [40]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf

# ====== CELL 1: PATHS & HYPERPARAMS ======

ROOT = os.getcwd()   # <-- FIX HERE

DATA_DIR = os.path.join(ROOT, "dataset")
MODEL_DIR = os.path.join(ROOT, "model")
EXPORT_DIR = os.path.join(ROOT, "export")

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS_HEAD = 15
EPOCHS_FT = 15
LR_HEAD = 1e-3
LR_FT = 1e-4

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f"Root directory: {ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Export directory: {EXPORT_DIR}")

Root directory: d:\Sinhala-Word-Recognizer
Data directory: d:\Sinhala-Word-Recognizer\dataset
Model directory: d:\Sinhala-Word-Recognizer\model
Export directory: d:\Sinhala-Word-Recognizer\export


# Data Preprocessing and Dataset Creation

In [41]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

class_names = train_ds.class_names
print("Classes:", class_names)
print("Number of classes:", len(class_names))


Found 603 files belonging to 314 classes.
Using 483 files for training.
Found 603 files belonging to 314 classes.
Using 120 files for validation.
Classes: ['අ', 'ආ', 'ඇ', 'ඈ', 'ඉ', 'ඊ', 'උ', 'ඌ', 'එ', 'ඒ', 'ඔ', 'ඕ', 'ක', 'ක්', 'කා', 'කැ', 'කෑ', 'කි', 'කී', 'කු', 'කූ', 'කෙ', 'කේ', 'කො', 'කෝ', 'ග', 'ග්', 'ගා', 'ගැ', 'ගෑ', 'ගි', 'ගී', 'ගු', 'ගූ', 'ගෙ', 'ගේ', 'ගො', 'ගෝ', 'ඟ', 'ඟා', 'ඟි', 'ඟී', 'ඟු', 'ඟූ', 'ඟෙ', 'ඟේ', 'ඟො', 'ඟෝ', 'ච', 'ච්', 'චා', 'චැ', 'චෑ', 'චි', 'චී', 'චු', 'චූ', 'චෙ', 'චේ', 'චො', 'චෝ', 'ජ', 'ජ්', 'ජා', 'ජැ', 'ජෑ', 'ජි', 'ජී', 'ජු', 'ජූ', 'ජෙ', 'ජේ', 'ජෝ', 'ට', 'ට්', 'ටා', 'ටැ', 'ටෑ', 'ටි', 'ටී', 'ටු', 'ටූ', 'ටෙ', 'ටේ', 'ටො', 'ටෝ', 'ඩ', 'ඩ්', 'ඩා', 'ඩැ', 'ඩෑ', 'ඩි', 'ඩී', 'ඩු', 'ඩූ', 'ඩෙ', 'ඩේ', 'ඩො', 'ඩෝ', 'ණ', 'ණ්', 'ණා', 'ණැ', 'ණෑ', 'ණි', 'ණී', 'ණු', 'ණූ', 'ණෙ', 'ණේ', 'ණො', 'ණෝ', 'ඬ', 'ඬා', 'ඬැ', 'ඬෑ', 'ඬි', 'ඬී', 'ඬු', 'ඬූ', 'ඬෙ', 'ඬේ', 'ඬො', 'ඬෝ', 'ත', 'ත්', 'තා', 'තැ', 'තෑ', 'ති', 'තී', 'තු', 'තූ', 'තෙ', 'තේ', 'තො', 'තෝ', 'ද', 'ද්', 'දා', 'දැ', 'දෑ', 'දි', 'දී', 'දු

In [42]:
AUTOTUNE = tf.data.AUTOTUNE

# Load dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Number of classes:", NUM_CLASSES)

# Grayscale → RGB (MobileNet needs 3 channels)
def gray_to_rgb(x, y):
    x = tf.image.grayscale_to_rgb(x)
    return x, y

train_ds = train_ds.map(gray_to_rgb).cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.map(gray_to_rgb).cache().prefetch(AUTOTUNE)

Found 603 files belonging to 314 classes.
Using 483 files for training.
Found 603 files belonging to 314 classes.
Using 120 files for validation.
Classes: ['අ', 'ආ', 'ඇ', 'ඈ', 'ඉ', 'ඊ', 'උ', 'ඌ', 'එ', 'ඒ', 'ඔ', 'ඕ', 'ක', 'ක්', 'කා', 'කැ', 'කෑ', 'කි', 'කී', 'කු', 'කූ', 'කෙ', 'කේ', 'කො', 'කෝ', 'ග', 'ග්', 'ගා', 'ගැ', 'ගෑ', 'ගි', 'ගී', 'ගු', 'ගූ', 'ගෙ', 'ගේ', 'ගො', 'ගෝ', 'ඟ', 'ඟා', 'ඟි', 'ඟී', 'ඟු', 'ඟූ', 'ඟෙ', 'ඟේ', 'ඟො', 'ඟෝ', 'ච', 'ච්', 'චා', 'චැ', 'චෑ', 'චි', 'චී', 'චු', 'චූ', 'චෙ', 'චේ', 'චො', 'චෝ', 'ජ', 'ජ්', 'ජා', 'ජැ', 'ජෑ', 'ජි', 'ජී', 'ජු', 'ජූ', 'ජෙ', 'ජේ', 'ජෝ', 'ට', 'ට්', 'ටා', 'ටැ', 'ටෑ', 'ටි', 'ටී', 'ටු', 'ටූ', 'ටෙ', 'ටේ', 'ටො', 'ටෝ', 'ඩ', 'ඩ්', 'ඩා', 'ඩැ', 'ඩෑ', 'ඩි', 'ඩී', 'ඩු', 'ඩූ', 'ඩෙ', 'ඩේ', 'ඩො', 'ඩෝ', 'ණ', 'ණ්', 'ණා', 'ණැ', 'ණෑ', 'ණි', 'ණී', 'ණු', 'ණූ', 'ණෙ', 'ණේ', 'ණො', 'ණෝ', 'ඬ', 'ඬා', 'ඬැ', 'ඬෑ', 'ඬි', 'ඬී', 'ඬු', 'ඬූ', 'ඬෙ', 'ඬේ', 'ඬො', 'ඬෝ', 'ත', 'ත්', 'තා', 'තැ', 'තෑ', 'ති', 'තී', 'තු', 'තූ', 'තෙ', 'තේ', 'තො', 'තෝ', 'ද', 'ද්', 'දා', 'දැ', 'දෑ', 'දි', 'දී', 'දු

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomTranslation(0.15, 0.15),
    tf.keras.layers.RandomZoom(0.15),
])



In [44]:
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False


c:\Users\Ravidu Senavirathna\.conda\envs\sinhala_cnn\lib\site-packages\keras\src\applications\mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


In [45]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)

x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_3 (Sequential)       │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 4, 4, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 576)            │         2,304 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 314)            │        40,506 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,055,786 (4.03 MB)

 Trainable params: 115,514 (451.23 KB)

 Non-trainable params: 940,272 (3.59 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_HEAD),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD
)

base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FT),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT
)



# Unfreeze last 30 layers only
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25
)


for layer in base_model.layers[:-20]:
    layer.trainable = False
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30
)



Epoch 1/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 214ms/step - accuracy: 0.0497 - loss: 5.6152 - val_accuracy: 0.1500 - val_loss: 5.3650
Epoch 2/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - accuracy: 0.2422 - loss: 4.5217 - val_accuracy: 0.1583 - val_loss: 5.0583
Epoch 3/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 109ms/step - accuracy: 0.2692 - loss: 3.8586 - val_accuracy: 0.1833 - val_loss: 4.9006
Epoch 4/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.2919 - loss: 3.5282 - val_accuracy: 0.2167 - val_loss: 4.8320
Epoch 5/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.3043 - loss: 3.2867 - val_accuracy: 0.2250 - val_loss: 4.8028
Epoch 6/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - accuracy: 0.3478 - loss: 2.9977 - val_accuracy: 0.2333 - val_loss: 4.7664
Epoch 7/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.3789 - loss: 2.7845 - val_accuracy: 0.2667 - val_loss: 4.7528
Epoch 8/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - accuracy: 0.4017 - loss: 2.5468 - val_accuracy: 0.2583 

In [ ]:
model.save(os.path.join(MODEL_DIR, "sinhala_letters.keras"))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open(os.path.join(EXPORT_DIR, "sinhala_letters.tflite"), "wb") as f:
    f.write(tflite_model)

print("✅ TFLite export completed")


INFO:tensorflow:Assets written to: C:\Users\RAVIDU~1\AppData\Local\Temp\tmpq4qhev3v\assets


INFO:tensorflow:Assets written to: C:\Users\RAVIDU~1\AppData\Local\Temp\tmpq4qhev3v\assets


Saved artifact at 'C:\Users\RAVIDU~1\AppData\Local\Temp\tmpq4qhev3v'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_175')
Output Type:
  TensorSpec(shape=(None, 314), dtype=tf.float32, name=None)
Captures:
  2115738574112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115738578512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115738578160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115738577280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115738572000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115738573232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115734826752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115734820768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115734826224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2115734827456: TensorSpec(shape=(), dtype=tf.resource, name=None

Save Class Labels

In [ ]:
import json

labels_path = os.path.join(EXPORT_DIR, "labels.json")

with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print("✅ Labels saved to:", labels_path)


✅ Labels saved to: d:\Sinhala-Word-Recognizer\export\labels.json


In [ ]:
# Testing the model
from PIL import Image
import numpy as np

img_path = "dataset/අ/අ.png"  # change to a real image

img = Image.open(img_path).convert("L").resize((128, 128))
img = np.array(img) / 255.0
img = np.expand_dims(img, axis=(0, -1))  # (1,128,128,1)
img = np.repeat(img, 3, axis=-1)          # RGB

pred = model.predict(img)
pred_idx = np.argmax(pred)
print("Predicted letter:", class_names[pred_idx])


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 588ms/step
Predicted letter: ග


In [ ]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(
    model_path=os.path.join(EXPORT_DIR, "sinhala_letters.tflite")
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

interpreter.set_tensor(input_details[0]["index"], img.astype(np.float32))
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]["index"])
print("TFLite prediction:", class_names[np.argmax(output)])


TFLite prediction: ග


c:\Users\Ravidu Senavirathna\.conda\envs\sinhala_cnn\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
# Checking Validation Accuracy
loss, acc = model.evaluate(val_ds)
print(f"Validation Accuracy: {acc*100:.2f}%")


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3667 - loss: 9.7209
Validation Accuracy: 36.67%
